# GNN scheduler — training notebook

Trains a pure-PyTorch graph neural network (GraphSAGE-style, no PyTorch Geometric) to imitate HEFT + HEFT-DS scheduling decisions on the 200-workflow corpus.

**Input:** `sim_res/training/*.csv` + `*.meta.json` generated by `BatchTrainingRunner`. Each meta JSON must include `workflow_xml_path` (added by the patched `TrainingLogger`).

**Output:** `simulator/src/main/resources/models/gnn_scheduler.onnx` — loaded at runtime by `GnnScheduler.java` via ONNX Runtime.

**Architecture:**
- Workflow encoder: 3 GraphSAGE layers over the DAG (task nodes + parent/child edges).
- Cluster encoder: 3 GraphSAGE layers over the 20-node cluster (geo-weighted edges).
- Cross-attention scorer: current-task embedding → score per cluster node.

In [1]:
import sys
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset, random_split

# Local imports from the same notebooks/ dir
sys.path.insert(0, str(Path('.').resolve()))
from gnn_data_pipeline import (
    NUM_CLUSTER_NODES,
    GraphSample,
    collate_graph_batch,
    load_training_samples,
)
from gnn_model import GnnScheduler, GnnSchedulerInference

REPO_ROOT  = Path('..').resolve()
DATA_DIR   = REPO_ROOT / 'sim_res' / 'training'
MODEL_OUT  = REPO_ROOT / 'src' / 'main' / 'resources' / 'models' / 'gnn_scheduler.onnx'
MODEL_OUT.parent.mkdir(parents=True, exist_ok=True)

DEVICE = 'mps' if torch.backends.mps.is_available() else ('cuda' if torch.cuda.is_available() else 'cpu')
print(f'device:  {DEVICE}')
print(f'data:    {DATA_DIR}')
print(f'output:  {MODEL_OUT}')

device:  mps
data:    /Users/busdavid/szakdoga/DISSECT-CF-Fog/simulator/sim_res/training
output:  /Users/busdavid/szakdoga/DISSECT-CF-Fog/simulator/src/main/resources/models/gnn_scheduler.onnx


## 1. Load training samples

Each CSV row in `sim_res/training/` becomes one graph sample (workflow DAG + cluster topology + current-task pointer + chosen-node label).

Only HEFT and HEFT-DS runs are used as teachers — the MaxMin / Adaptive round-robin teachers' decisions are not state-deterministic and confuse the model (lesson from the earlier MLP iteration).

In [2]:
samples = load_training_samples(
    DATA_DIR,
    keep_schedulers=('heft', 'heftds'),
    verbose=True,
)
print(f'\ntotal samples: {len(samples)}')

# Sanity prints
if samples:
    s = samples[0]
    print(f'first sample shapes:')
    print(f'  wf_node_features: {s.wf_node_features.shape}')
    print(f'  wf_adj_norm:      {s.wf_adj_norm.shape}')
    print(f'  cl_node_features: {s.cl_node_features.shape}')
    print(f'  cl_adj_norm:      {s.cl_adj_norm.shape}')
    print(f'  current_task_idx: {s.current_task_idx}')
    print(f'  label:            {s.label}')
    print(f'  sample_weight:    {s.sample_weight:.6f}')

# Label distribution
labels = np.array([s.label for s in samples])
print('\nlabels per cluster node:')
for i in range(NUM_CLUSTER_NODES):
    print(f'  node{i:2d}: {(labels == i).sum():5d}')

# Workflow diversity
wf_names = set(s.workflow_name for s in samples)
print(f'\ndistinct workflows in dataset: {len(wf_names)}')

scanning 1336 sidecar JSONs in /Users/busdavid/szakdoga/DISSECT-CF-Fog/simulator/sim_res/training
loaded 164254 graph samples
  ⚠ 206 sidecars had no workflow_xml_path or missing XML

total samples: 164254
first sample shapes:
  wf_node_features: (100, 6)
  wf_adj_norm:      (100, 100)
  cl_node_features: (20, 8)
  cl_adj_norm:      (20, 20)
  current_task_idx: 2
  label:            11
  sample_weight:    0.001093

labels per cluster node:
  node 0: 18292
  node 1:  7150
  node 2: 12388
  node 3:  6136
  node 4: 11050
  node 5:  5590
  node 6:  5403
  node 7: 10583
  node 8:  5275
  node 9:  5055
  node10:  4823
  node11: 14945
  node12:  4879
  node13:  4772
  node14:  4873
  node15:  9637
  node16:  4672
  node17:  4422
  node18:  9580
  node19: 14729

distinct workflows in dataset: 231


## 2. Train / val split (by source_run)

We split BY WORKFLOW RUN (not by row) so the validation set contains entire runs the model never saw during training. Same approach as the MLP notebook.

In [3]:
rng = np.random.default_rng(42)
unique_runs = sorted(set(s.source_run for s in samples))
rng.shuffle(unique_runs)
n_val   = max(1, int(0.15 * len(unique_runs)))
val_run_set   = set(unique_runs[:n_val])
train_run_set = set(unique_runs[n_val:])

train_samples = [s for s in samples if s.source_run in train_run_set]
val_samples   = [s for s in samples if s.source_run in val_run_set]
print(f'runs : train={len(train_run_set)}  val={len(val_run_set)}')
print(f'rows : train={len(train_samples)}  val={len(val_samples)}')

runs : train=393  val=69
rows : train=140038  val=24216


## 3. DataLoaders

In [4]:
class _ListDataset(Dataset):
    def __init__(self, items): self.items = items
    def __len__(self):         return len(self.items)
    def __getitem__(self, i):  return self.items[i]

BATCH = 16
train_loader = DataLoader(_ListDataset(train_samples), batch_size=BATCH, shuffle=True,
                          collate_fn=collate_graph_batch, num_workers=0)
val_loader   = DataLoader(_ListDataset(val_samples),   batch_size=BATCH, shuffle=False,
                          collate_fn=collate_graph_batch, num_workers=0)
print(f'batches: train={len(train_loader)}  val={len(val_loader)}')

batches: train=8753  val=1514


## 4. Model + optimizer

In [5]:
model = GnnScheduler(
    wf_in_dim=6, cl_in_dim=8, hidden_dim=64, num_layers=3
).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(model)
print(f'parameters: {n_params:,}')

# Lower learning rate than the MLP run — the model converged in ~15 epochs at lr=1e-3,
# which suggests it was overshooting the local minimum. 5e-4 lets it refine longer.
opt = torch.optim.AdamW(model.parameters(), lr=5e-4, weight_decay=1e-4)
ce  = nn.CrossEntropyLoss(reduction='none')

# ---- inverse-frequency class weights for the 20 cluster nodes ----
# The HEFT/HEFT-DS labels in our corpus are cloud-heavy (node0/11/19 dominate)
# and the rare fog nodes (3, 8, 10, 12, 13, 14) only get a handful of samples.
# Without class weights the first GNN run collapsed those classes to ~15% acc.
# Apply 1/freq weighting (normalized) so each class contributes ~equally to the loss.
all_labels = np.array([s.label for s in train_samples], dtype=np.int64)
counts = np.bincount(all_labels, minlength=NUM_CLUSTER_NODES).astype(np.float32)
freq   = counts / max(counts.sum(), 1.0)
freq[freq == 0] = 1.0
class_w = 1.0 / freq
class_w = class_w / class_w.mean()
class_weight_tensor = torch.from_numpy(class_w.astype(np.float32)).to(DEVICE)
print('\nclass weights:')
for i, w in enumerate(class_w):
    print(f'  node{i:2d}  count={int(counts[i]):5d}  weight={w:.3f}')

GnnScheduler(
  (wf_layers): ModuleList(
    (0): GraphSageBlock(
      (lin_self): Linear(in_features=6, out_features=64, bias=True)
      (lin_nbr): Linear(in_features=6, out_features=64, bias=True)
    )
    (1-2): 2 x GraphSageBlock(
      (lin_self): Linear(in_features=64, out_features=64, bias=True)
      (lin_nbr): Linear(in_features=64, out_features=64, bias=True)
    )
  )
  (cl_layers): ModuleList(
    (0): GraphSageBlock(
      (lin_self): Linear(in_features=8, out_features=64, bias=True)
      (lin_nbr): Linear(in_features=8, out_features=64, bias=True)
    )
    (1-2): 2 x GraphSageBlock(
      (lin_self): Linear(in_features=64, out_features=64, bias=True)
      (lin_nbr): Linear(in_features=64, out_features=64, bias=True)
    )
  )
  (query_proj): Linear(in_features=64, out_features=64, bias=True)
  (key_proj): Linear(in_features=64, out_features=64, bias=True)
  (score_head): Linear(in_features=64, out_features=1, bias=True)
  (dropout): Dropout(p=0.1, inplace=False)
)
p

## 5. Training loop with early stopping

In [6]:
EPOCHS   = 200
PATIENCE = 15

def run_epoch(loader, train: bool):
    model.train() if train else model.eval()
    total_loss = 0.0
    correct    = 0
    seen       = 0
    cm = torch.zeros((NUM_CLUSTER_NODES, NUM_CLUSTER_NODES), dtype=torch.long)
    grad_ctx = torch.enable_grad() if train else torch.no_grad()
    with grad_ctx:
        for batch in loader:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            logits = model(
                batch['wf_node_features'],
                batch['wf_adj_norm'],
                batch['cl_node_features'],
                batch['cl_adj_norm'],
                batch['current_task_idx'],
            )
            # combined weight = reward (1/makespan) * class_weight[label]
            per_class_w = class_weight_tensor[batch['labels']]
            per_sample  = ce(logits, batch['labels']) * batch['sample_weights'] * per_class_w
            loss = per_sample.mean()
            if train:
                opt.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
                opt.step()
            bs = batch['labels'].size(0)
            total_loss += loss.item() * bs
            preds = logits.argmax(dim=-1)
            correct += (preds == batch['labels']).sum().item()
            seen    += bs
            if not train:
                for t, p in zip(batch['labels'].cpu(), preds.cpu()):
                    cm[t, p] += 1
    return total_loss / max(1, seen), correct / max(1, seen), cm

best_val = float('inf')
best_state = None
stale = 0
history = []

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc, _ = run_epoch(train_loader, train=True)
    va_loss, va_acc, _ = run_epoch(val_loader,   train=False)
    history.append({'epoch': epoch, 'train_loss': tr_loss, 'train_acc': tr_acc,
                    'val_loss': va_loss, 'val_acc': va_acc})
    flag = ''
    if va_loss < best_val - 1e-4:
        best_val   = va_loss
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        stale = 0; flag = ' ✓'
    else:
        stale += 1
    print(f'epoch {epoch:3d}  train loss={tr_loss:.4f} acc={tr_acc:.3f}   val loss={va_loss:.4f} acc={va_acc:.3f}{flag}')
    if stale >= PATIENCE:
        print(f'early stop @ epoch {epoch} (no val improvement in {PATIENCE} epochs)')
        break

if best_state is not None:
    model.load_state_dict(best_state)
print(f'\nbest val loss: {best_val:.4f}')

epoch   1  train loss=0.0026 acc=0.240   val loss=0.0025 acc=0.389 ✓
epoch   2  train loss=0.0024 acc=0.383   val loss=0.0024 acc=0.424
epoch   3  train loss=0.0023 acc=0.404   val loss=0.0024 acc=0.422 ✓
epoch   4  train loss=0.0023 acc=0.412   val loss=0.0023 acc=0.444
epoch   5  train loss=0.0023 acc=0.418   val loss=0.0023 acc=0.457
epoch   6  train loss=0.0023 acc=0.426   val loss=0.0023 acc=0.443
epoch   7  train loss=0.0023 acc=0.430   val loss=0.0023 acc=0.441
epoch   8  train loss=0.0023 acc=0.435   val loss=0.0023 acc=0.459
epoch   9  train loss=0.0023 acc=0.436   val loss=0.0023 acc=0.454
epoch  10  train loss=0.0023 acc=0.441   val loss=0.0023 acc=0.466
epoch  11  train loss=0.0022 acc=0.443   val loss=0.0023 acc=0.460
epoch  12  train loss=0.0022 acc=0.446   val loss=0.0023 acc=0.468
epoch  13  train loss=0.0022 acc=0.448   val loss=0.0023 acc=0.465
epoch  14  train loss=0.0022 acc=0.450   val loss=0.0023 acc=0.463
epoch  15  train loss=0.0022 acc=0.453   val loss=0.0023 a

## 6. Per-class accuracy + confusion matrix on val

In [7]:
_, va_acc, cm = run_epoch(val_loader, train=False)
print(f'final val accuracy: {va_acc:.3f}\n')
print('per-node val accuracy (only nodes that appear in val):')
for n in range(NUM_CLUSTER_NODES):
    n_true = cm[n].sum().item()
    if n_true == 0:
        continue
    acc = cm[n, n].item() / n_true
    print(f'  node{n:2d}  n={n_true:5d}  acc={acc:.3f}')

from collections import Counter
miscls = Counter()
for t in range(NUM_CLUSTER_NODES):
    for p in range(NUM_CLUSTER_NODES):
        if t != p and cm[t, p].item() > 0:
            miscls[(t, p)] += int(cm[t, p].item())
print('\ntop 10 confusion pairs (true → pred):')
for (t, p), c in miscls.most_common(10):
    print(f'  node{t:2d} → node{p:2d}   {c}')

final val accuracy: 0.422

per-node val accuracy (only nodes that appear in val):
  node 0  n= 2774  acc=0.545
  node 1  n=  993  acc=0.369
  node 2  n= 1749  acc=0.435
  node 3  n=  884  acc=0.252
  node 4  n= 1518  acc=0.550
  node 5  n=  895  acc=0.361
  node 6  n=  793  acc=0.367
  node 7  n= 1417  acc=0.536
  node 8  n=  793  acc=0.154
  node 9  n=  785  acc=0.205
  node10  n=  742  acc=0.318
  node11  n= 2298  acc=0.453
  node12  n=  777  acc=0.261
  node13  n=  750  acc=0.268
  node14  n=  747  acc=0.316
  node15  n= 1306  acc=0.409
  node16  n=  728  acc=0.493
  node17  n=  684  acc=0.320
  node18  n= 1336  acc=0.494
  node19  n= 2247  acc=0.520

top 10 confusion pairs (true → pred):
  node11 → node19   294
  node 2 → node 0   279
  node 0 → node19   260
  node11 → node 0   256
  node 0 → node 6   250
  node19 → node 6   243
  node11 → node 6   241
  node19 → node 0   231
  node 2 → node 4   182
  node15 → node18   160


## 7. ONNX export

We wrap the model in `GnnSchedulerInference` (un-batched single-sample input) and export with **dynamic axes** for `N_tasks` and `N_nodes`, so the same ONNX file can serve any workflow / cluster size. The Java side passes one task at a time.

In [8]:
wrapped = GnnSchedulerInference(model).to('cpu').eval()

# Dummy single-sample inputs with the largest sizes we expect at runtime.
# Dynamic axes will let smaller sizes work too at inference time.
dummy_n_t = 64                   # typical mid-size workflow
dummy_n_n = NUM_CLUSTER_NODES
wf_feat = torch.zeros(dummy_n_t, 6,        dtype=torch.float32)
wf_adj  = torch.zeros(dummy_n_t, dummy_n_t, dtype=torch.float32)
cl_feat = torch.zeros(dummy_n_n, 8,        dtype=torch.float32)
cl_adj  = torch.zeros(dummy_n_n, dummy_n_n, dtype=torch.float32)
cur_idx = torch.zeros(1,                   dtype=torch.int64)

torch.onnx.export(
    wrapped,
    (wf_feat, wf_adj, cl_feat, cl_adj, cur_idx),
    MODEL_OUT.as_posix(),
    input_names=[
        'wf_node_features',
        'wf_adj_norm',
        'cl_node_features',
        'cl_adj_norm',
        'current_task_idx',
    ],
    output_names=['node_scores'],
    dynamic_axes={
        'wf_node_features': {0: 'N_tasks'},
        'wf_adj_norm':      {0: 'N_tasks', 1: 'N_tasks'},
        'cl_node_features': {0: 'N_nodes'},
        'cl_adj_norm':      {0: 'N_nodes', 1: 'N_nodes'},
        'node_scores':      {0: 'N_nodes'},
    },
    opset_version=17,
)
size_kb = MODEL_OUT.stat().st_size / 1024
print(f'✓ exported {MODEL_OUT}  ({size_kb:.1f} KB)')

# Sanity check: reload via onnxruntime, run inference, compare to PyTorch
try:
    import onnxruntime as ort
    sess = ort.InferenceSession(MODEL_OUT.as_posix())
    if val_samples:
        s = val_samples[0]
        ort_inputs = {
            'wf_node_features': s.wf_node_features.astype(np.float32),
            'wf_adj_norm':      s.wf_adj_norm.astype(np.float32),
            'cl_node_features': s.cl_node_features.astype(np.float32),
            'cl_adj_norm':      s.cl_adj_norm.astype(np.float32),
            'current_task_idx': np.asarray([s.current_task_idx], dtype=np.int64),
        }
        onnx_out = sess.run(['node_scores'], ort_inputs)[0]

        torch_inputs = {k: torch.from_numpy(v) for k, v in ort_inputs.items()}
        torch_out = wrapped(
            torch_inputs['wf_node_features'],
            torch_inputs['wf_adj_norm'],
            torch_inputs['cl_node_features'],
            torch_inputs['cl_adj_norm'],
            torch_inputs['current_task_idx'],
        ).detach().numpy()
        max_diff = float(np.abs(onnx_out - torch_out).max())
        print(f'ONNX vs PyTorch max score diff: {max_diff:.2e}  (should be < 1e-4)')
        print(f'ONNX argmax: {int(onnx_out.argmax())}  PyTorch argmax: {int(torch_out.argmax())}  True label: {s.label}')
except ImportError:
    print('onnxruntime not installed; skipping cross-check (pip install onnxruntime)')

/var/folders/8_/xmwv32ks4p9d8jk8ddcqchfw0000gn/T/ipykernel_18968/226633254.py:13: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0515 03:04:20.271000 18968 site-packages/torch/onnx/_internal/exporter/_compat.py:133] Setting ONNX exporter to use operator set version 18 because the requested opset_version 17 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features
W0515 03:04:21.444000 18968 site-packages/torch/onnx/_internal/exporter/_registration.py:110] torchvision is not installed. Skipping torchvision::nms
W0515 03:04:21.

[torch.onnx] Obtain model graph for `GnnSchedulerInference([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `GnnSchedulerInference([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
The model version conversion is not supported by the onnxscript version converter and fallback is enabled. The model will be converted using the onnx C API (target version: 17).


[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
✓ exported /Users/busdavid/szakdoga/DISSECT-CF-Fog/simulator/src/main/resources/models/gnn_scheduler.onnx  (14.3 KB)
ONNX vs PyTorch max score diff: 2.86e-06  (should be < 1e-4)
ONNX argmax: 0  PyTorch argmax: 0  True label: 11


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/torch/onnx/_internal/exporter/_onnx_program.py:487: UserWarning: # The axis name: N_tasks will not be used, since it shares the same shape constraints with another axis: N_tasks.
  rename_mapping = _dynamic_shapes.create_rename_mapping(
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/torch/onnx/_internal/exporter/_onnx_program.py:487: UserWarning: # The axis name: N_nodes will not be used, since it shares the same shape constraints with another axis: N_nodes.
  rename_mapping = _dynamic_shapes.create_rename_mapping(


## Done

The `gnn_scheduler.onnx` file is now ready for the Java side. Next step: implement `GnnScheduler.java` and `GnnGraphBuilder.java`, then re-run the benchmark.